# AEMO impact-aware DT retrain (MoLab)

Retrains the **modern v2** Decision Transformer (8×768 GQA) on the
**impact-aware dataset** generated in Phase 4 (batteries 8/50/150/250 MWh,
horizons 12d/8wk/26wk/~9mo, real_world degradation, piecewise-merit-order
market impact).

The dataset lives on HuggingFace (uploaded manually). This notebook downloads
it, then runs `scripts/pretrain_aemo_decision_transformer.py` with the
modern-v2 hyperparameters.

**Pipeline note:** this is a *pretrain* (offline), distinct from GRPO
post-training. After retraining, validate the impact-aware DT vs the
pretrained v2 on the Phase 3 surface.

In [ ]:
# Repo bootstrap (run this notebook from the energydecision repo root)
from pathlib import Path
import os, sys

def _find_repo_root(start: Path) -> Path:
    for cand in (start, *start.parents):
        if (cand / "README.md").exists() and (cand / "src").exists():
            return cand
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("repo root:", REPO_ROOT)

## 1. Download the impact-aware dataset from HuggingFace

Set `HF_REPO` to the repo you uploaded to (e.g. `mrvictoru/AEMO_simulated_impact_trade`)
and `HF_FILENAME` to the assembled parquet (default `aemo_impact_dataset.parquet`).

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO = "mrvictoru/AEMO_simulated_impact_trade"  # <-- your uploaded repo
HF_FILENAME = "aemo_impact_dataset.parquet"

dataset_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILENAME)
print("dataset downloaded to:", dataset_path)

import polars as pl
d = pl.read_parquet(dataset_path)
print("rows:", d.height, "episodes:", d['episode_id'].n_unique())
print("sources:", d.group_by('source_policy').len().sort('source_policy'))

## 2. Model config + training hyperparameters (modern v2)

In [ ]:
from pathlib import Path
import json

MODEL_CONFIG_PATH = str(REPO_ROOT / "configs" / "aemo_decision_transformer_model_kwargs_modern_v2_full_fcas.json")
print("model config:", json.dumps(json.load(open(MODEL_CONFIG_PATH)), indent=2)[:500])

DT_TRAINING_ARGS = dict(
    epochs=2,
    batch_size=64,
    lr=3e-5,
    val_split=0.1,
    seed=42,
    amp_mode='bf16',          # MoLab GPU; use 'fp16' on older GPUs
    return_scale=1.0,         # modern v2 (checkpoint return_scale)
    action_loss_weight=0.999,
    state_loss_weight=0.002,
    return_loss_weight=0.0001,
    weight_decay=0.01,
    num_workers=4,
    prefetch_factor=2,
)
print("training args:", DT_TRAINING_ARGS)

## 3. Build + run the pretrain command

Outputs (model, checkpoint, loss history) go to `data/aemo_dt_impact/retrain/`.

In [ ]:
out_dir = REPO_ROOT / "data" / "aemo_dt_impact" / "retrain"
out_dir.mkdir(parents=True, exist_ok=True)

command = [
    'python3',
    str(REPO_ROOT / 'scripts' / 'pretrain_aemo_decision_transformer.py'),
    '--dataset-path', str(dataset_path),
    '--model-config', MODEL_CONFIG_PATH,
    '--save-path', str(out_dir / 'aemo_dt_impact_model.pt'),
    '--checkpoint-path', str(out_dir / 'aemo_dt_impact_checkpoint.pt'),
    '--loss-csv-path', str(out_dir / 'aemo_dt_loss_history.csv'),
    '--epochs', str(DT_TRAINING_ARGS['epochs']),
    '--batch-size', str(DT_TRAINING_ARGS['batch_size']),
    '--lr', str(DT_TRAINING_ARGS['lr']),
    '--val-split', str(DT_TRAINING_ARGS['val_split']),
    '--seed', str(DT_TRAINING_ARGS['seed']),
    '--amp-mode', DT_TRAINING_ARGS['amp_mode'],
    '--return-scale', str(DT_TRAINING_ARGS['return_scale']),
    '--action-loss-weight', str(DT_TRAINING_ARGS['action_loss_weight']),
    '--state-loss-weight', str(DT_TRAINING_ARGS['state_loss_weight']),
    '--return-loss-weight', str(DT_TRAINING_ARGS['return_loss_weight']),
    '--weight-decay', str(DT_TRAINING_ARGS['weight_decay']),
    '--num-workers', str(DT_TRAINING_ARGS['num_workers']),
    '--prefetch-factor', str(DT_TRAINING_ARGS['prefetch_factor']),
]
print('AEMO impact DT training command:')
print(' '.join(command))

In [ ]:
import subprocess
print("running training ...")
proc = subprocess.run(command, cwd=str(REPO_ROOT))
print("training finished, return code:", proc.returncode)

## 4. Next
- Validate the impact-aware DT on the Phase 3 surface:
  `python3 scripts/phase3_impact_eval.py` (points at `models/aemo/dt/hf_v2_modern/` —
  copy the new checkpoint there, or edit the script's checkpoint path).
- Compare vs the pretrained modern v2 (identity + impact) and against Oracle_MI.